In [0]:
from pyspark.sql.functions import max as _max
import pyspark.sql.functions as F
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, LongType

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("run_type", "job_run")


run_id = dbutils.widgets.get("run_id")
job_id = dbutils.widgets.get("job_id")
run_type = dbutils.widgets.get("run_type")

In [0]:
print(run_id)
print(job_id)

In [0]:
full_table_name = 'databricksformula1.default.table_runs'
def get_max_run_id():
    try:
        df = spark.read.table(full_table_name)
        max_run_id = df.select(_max('run_id').alias('run_id')).collect()[0]['run_id']
        max_run_id = max_run_id + 1
        return max_run_id
    except Exception as e:
        print(e)
        return 1

def get_max_job_id():
    try:
        df = spark.read.table(full_table_name)
        max_job_id = df.select(_max('job_id').alias('job_id')).collect()[0]['job_id']
        max_job_id = max_job_id + 1
        return max_job_id
    except:
        return 1

if run_type == "manual" or (not job_id and not run_id):
    run_id = get_max_run_id()
    job_id = get_max_job_id()

print(run_id)
print(job_id)



In [0]:
%sql
CREATE table if not exists databricksformula1.default.table_runs (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    run_id BIGINT,
    job_id BIGINT,
    type_of_run VARCHAR(255),
    created_timestamp TIMESTAMP
);


In [0]:
test_schema = StructType([
    StructField("run_id", LongType(), True),
    StructField("job_id", LongType(), True),
    StructField("type_of_run", StringType(), True),
    StructField("created_timestamp", TimestampType(), True)
])

In [0]:
df = spark.createDataFrame([(int(run_id), int(job_id), run_type, datetime.now())], schema=test_schema)
df.display()
df.write.mode('append').saveAsTable('databricksformula1.default.table_runs')